In [1]:
import os
import numpy as np
import tifffile

from google.colab import drive
drive.mount('/content/drive')

# Navigate to your folder where the TIFF files are located
os.chdir("/content/drive/MyDrive/Colab Notebooks")

# Your TIFF files
tiff_files = [
    "F+B+lambda first 100 um top cornea 10X.tif",
    "F+B+Lambda starting from 500 um cornea 10x.tif",
    "Forward_backward_lambda_scan_lenticule.tif",
    "SMILE 160 um - around the SMILE-  25 mmHg deflated to 23 (5 um interval).tif",
    "SMILE 160 um - 50un right below the plane of the SMILE 25 mmHg deflated to 23 (0.81 um interval).tif",
    "SMILE 160 um - 50un right below the plane of the SMILE 14 mmHg deflated to 14 (0.81 um interval).tif",
    "SMILE 160 um - 50un right below the plane of the SMILE 10 mmHg deflated to 8 (0.81 um interval).tif"
]

def process_tiff_files(file_list):
    """
    Reads TIFF files, extracts backward and forward slices, and returns them.

    Args:
        file_list (list): A list of TIFF file paths to process.

    Returns:
        tuple: A tuple containing two lists:
               - backward_slices (list of np.array): Extracted backward channel slices.
               - forward_slices (list of np.array): Extracted forward channel slices.
    """
    all_backward_slices = []
    all_forward_slices = []
    total_slice_pairs = 0

    for file_path in file_list:
        print(f"\nReading {file_path} ...")
        img = tifffile.imread(file_path)
        print(f"Original shape: {img.shape}")

        img = np.squeeze(img)  # Remove singleton dimensions
        if img.ndim != 4:
            raise ValueError(f"{file_path} should have shape (Z, C, Y, X), got {img.shape}")

        n_stacks, n_channels, Y, X = img.shape
        print(f"{n_stacks} stacks × {n_channels} channels ({Y}×{X})")

        # Extract backward and forward slices for each stack
        for z in range(n_stacks):
            # Assuming channel 1 for backward and channel 4 for forward based on original code
            all_backward_slices.append(img[z, 1, :, :])  # channel 2 (index 1)
            all_forward_slices.append(img[z, 4, :, :])   # channel 5 (index 4)
            total_slice_pairs += 1

    print(f"\n Extracted {total_slice_pairs} slice pairs across all files.")
    print(f"backward_slices list length: {len(all_backward_slices)}")
    print(f"forward_slices list length: {len(all_forward_slices)}")

    return all_backward_slices, all_forward_slices

for file_path in tiff_files:
    backward_slices, forward_slices = process_tiff_files([file_path])
    # then extract patches and save for this file individually

    # Extract 256×256 patches
    bk_patches = []
    fw_patches = []
    for bk, fw in zip(backward_slices, forward_slices):
        h, w = bk.shape
        for y in range(0, h - 255, 256):
            for x in range(0, w - 255, 256):
                bk_patches.append(bk[y:y+256, x:x+256])
                fw_patches.append(fw[y:y+256, x:x+256])


    # Normalise both channels the same way
    bk_arr = np.stack(bk_patches).astype(np.float32)
    fw_arr = np.stack(fw_patches).astype(np.float32)
    bk_arr = (bk_arr - bk_arr.min()) / (bk_arr.max() - bk_arr.min() + 1e-8)
    fw_arr = (fw_arr - fw_arr.min()) / (fw_arr.max() - fw_arr.min() + 1e-8)


    # Save with filename derived from the tif
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    np.save(f'backward_{file_name}.npy', bk_arr)
    np.save(f'forward_{file_name}.npy', fw_arr)
    print(f'Saved {len(bk_patches)} patches for {file_name}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Reading F+B+lambda first 100 um top cornea 10X.tif ...
Original shape: (28, 5, 1024, 1024)
28 stacks × 5 channels (1024×1024)

 Extracted 28 slice pairs across all files.
backward_slices list length: 28
forward_slices list length: 28
Saved 448 patches for F+B+lambda first 100 um top cornea 10X

Reading F+B+Lambda starting from 500 um cornea 10x.tif ...
Original shape: (35, 5, 1024, 1024)
35 stacks × 5 channels (1024×1024)

 Extracted 35 slice pairs across all files.
backward_slices list length: 35
forward_slices list length: 35
Saved 560 patches for F+B+Lambda starting from 500 um cornea 10x

Reading Forward_backward_lambda_scan_lenticule.tif ...
Original shape: (193, 5, 1024, 1024)
193 stacks × 5 channels (1024×1024)

 Extracted 193 slice pairs across all files.
backward_slices list length: 193
forward_slices list length: 193
Saved 3088 patches for Forward_

In [ ]:

import numpy as np
#import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from skimage import exposure
from skimage.transform import resize # Import resize (kept for consistency, but not used for initial image processing)
from scipy.ndimage import median_filter

# This section prepares the image data for a machine learning model by converting raw slices
# into smaller, manageable patches and then normalizing and binarizing them.

# The variables `backward_slices` and `forward_slices` are assumed to be lists of NumPy arrays,
# where each array represents a 2D image slice (e.g., 1024x1024 pixels) obtained from the TIFF files.

# Patch extraction function definition:
def extract_patches(image_2d, patch_size, stride):
    """
    Extracts square patches from a 2D image.

    Args:
        image_2d (np.array): The input 2D NumPy array representing a single image slice.
        patch_size (int): The dimension of the square patches to extract (e.g., 256 for 256x256 patches).
        stride (int): The step size used to move the extraction window across the image.
                      If stride equals patch_size, patches are non-overlapping.

    Returns:
        list: A list of 2D NumPy arrays, where each array is an extracted patch.
    """
    H, W = image_2d.shape # Get the height and width of the input 2D image.
    patches = []         # Initialize an empty list to store the extracted patches.

    # Iterate over the height of the image with a step of 'stride'.
    # The loop stops when the patch would extend beyond the image boundary.
    for i in range(0, H - patch_size + 1, stride):
      # Iterate over the width of the image with a step of 'stride'.
      for j in range(0, W - patch_size + 1, stride):
        # Extract a square patch from the current (i, j) position.
        patch = image_2d[i:i+patch_size, j:j+patch_size]
        patches.append(patch) # Add the extracted patch to the list.

    return patches

# Define patch extraction parameters:
patch_size = 256 # Each patch will be 256x256 pixels.
stride = 256     # A stride equal to patch_size ensures that patches do not overlap.

bk_patches_list = [] # Temporary list to collect all backward channel patches.
# Loop through each 2D image slice in the `backward_slices` list.
for img_slice in backward_slices:
    # Extract patches from the current slice. Convert the slice to float type for numerical stability
    # and compatibility with subsequent processing (e.g., normalization, CLAHE).
    # `extend` is used to add all patches from `extract_patches` output directly to `bk_patches_list`.
    bk_patches_list.extend(extract_patches(img_slice.astype(float), patch_size, stride))

fw_patches_list = [] # Temporary list to collect all forward channel patches.
# Loop through each 2D image slice in the `forward_slices` list.
for img_slice in forward_slices:
    # Similar to backward slices, extract patches from forward slices, ensuring float type.
    fw_patches_list.extend(extract_patches(img_slice.astype(float), patch_size, stride))

# Convert the lists of 2D patches into single 3D NumPy arrays.
# `np.stack` takes a list of arrays and stacks them along a new axis,
# resulting in a shape of (number_of_patches, patch_height, patch_width).
backward_patches = np.stack(bk_patches_list)
forward_patches = np.stack(fw_patches_list)

print(f'Backward patches shape after stack: {backward_patches.shape}') # Display the dimensions of the stacked backward patches.
print(f'Forward patches shape after stack: {forward_patches.shape}')   # Display the dimensions of the stacked forward patches.

# Normalization step: Calculate global percentiles across all original slices.
# This approach uses overall data distribution for consistent normalization.

# Concatenate all pixels from all backward slices into a single 1D array.
all_pixels_b = np.concatenate([img_slice.flatten() for img_slice in backward_slices])
# Concatenate all pixels from all forward slices into a single 1D array.
all_pixels_f = np.concatenate([img_slice.flatten() for img_slice in forward_slices])

# Calculate the 1st and 99th percentiles for both backward and forward image pixel distributions.
# These percentiles define the range for intensity scaling, effectively clipping outliers.
p1_b, p99_b = np.percentile(all_pixels_b, (1, 99)) # p1_b = value below which 1% of pixels lie (dark threshold), p99_b = value below which 99% of pixels lie (bright threshold).
p1_f, p99_f = np.percentile(all_pixels_f, (1, 99))

# Normalization function definition:
def normalize_percentile(x, p1, p99):
    """
    Normalizes image pixel values using 1st and 99th percentiles for clipping and scaling.

    Args:
        x (np.array): Input image array (can be 2D or 3D, operates element-wise).
        p1 (float): The 1st percentile value (lower clipping limit).
        p99 (float): The 99th percentile value (upper clipping limit).

    Returns:
        np.array: Normalized image array with pixel values mapped to the range [0, 1].
    """
    # Scale pixel values: (x - p1) shifts the distribution so that the 1st percentile becomes 0.
    # Dividing by (p99 - p1) scales the range between p1 and p99 to [0, 1].
    x = (x - p1) / (p99 - p1)
    # Clip values: Any pixel value below 0 (originally less than p1) becomes 0.
    # Any pixel value above 1 (originally greater than p99) becomes 1.
    x = np.clip(x, 0, 1)
    # Convert the array to float32, which is a common data type for deep learning models.
    return x.astype(np.float32)

# Apply the percentile-based normalization to each extracted patch.
backward_norm = [
    normalize_percentile(p, p1_b, p99_b) for p in backward_patches
] # Each backward patch is normalized individually using the global backward percentiles.

forward_norm = [
    normalize_percentile(p, p1_f, p99_f) for p in forward_patches
] # Each forward patch is normalized individually using the global forward percentiles.




# Print the minimum and maximum pixel values to verify the outcome of normalization and binarization.
# For normalized images, values should be within [0, 1]. For binarized images, values should be 0 or 1.
print(f'backward min max: {np.min(backward_norm)}, {np.max(backward_norm)}')
print(f'forward min max: {np.min(forward_norm)}, {np.max(forward_norm)} ')

In [ ]:
# This cell visualizes the preprocessing steps for a single image.
# It helps to understand the effect of each operation on the image.

import matplotlib.pyplot as plt # Import Matplotlib for plotting.

indx = 4550 # Index of a specific patch to visualize.

# Reapply CLAHE (Contrast Limited Adaptive Histogram Equalization) to the selected normalized forward patch for demonstration.
clahe_forward = exposure.equalize_adapthist(forward_norm[indx], clip_limit=0.3)

# Apply thresholding to binarize the image.
thresholded = clahe_forward.copy()
thresholded[thresholded < 0.7] = 0 # Pixels below 0.7 become black.
thresholded[(thresholded >= 0.7) & (thresholded < 1)] = 1 # Pixels between 0.7 and 1 become white.

# Apply a median filter to the thresholded image to reduce noise.
median_filtered = median_filter(thresholded, size=2)

# Create a figure with 5 subplots to display the different stages of processing.
fig, axes = plt.subplots(1, 5, figsize=(16, 5))

# Display the raw (unnormalized) forward patch.
axes[0].imshow(forward_patches[indx], cmap='gray')
axes[0].set_title('Raw Forward')
axes[0].axis('off')

# Display the normalized forward patch.
axes[1].imshow(forward_norm[indx], cmap='gray')
axes[1].set_title('Normalised Forward')
axes[1].axis('off')

# Display the patch after CLAHE application.
axes[2].imshow(clahe_forward, cmap='gray')
axes[2].set_title('After CLAHE')
axes[2].axis('off')

# Display the patch after thresholding.
axes[3].imshow(thresholded, cmap='gray')
axes[3].set_title('After Threshold (t = 0.7)')
axes[3].axis('off')

# Display the patch after median filtering.
axes[4].imshow(median_filtered, cmap='gray')
axes[4].set_title('After Median Filter (Kernel = 3)') # Note: The code uses size=2, but the title says Kernel=3. This might be a discrepancy to check.
axes[4].axis('off')

# Save the generated figure to a file.
plt.savefig('preprocessing_pipeline.png', dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_erosion, binary_dilation
from scipy.ndimage import median_filter

#back = np.load('backwards_small.npy')
#forw = np.load('forwards_small.npy')

# filter kernel
struct = np.ones((2, 2), dtype=bool)  # 5x5 square kernel
print(struct)
# erosion and dilation
indx = 600
forw_erosion = binary_erosion(forw[indx], struct)
forw_dilation = binary_dilation(forw_erosion, struct)

# Get the kernel size directly from the median_filter call
median_filter_kernel_size = 3 # The size parameter used in the median_filter call
median_filtered_forw = median_filter(forw[indx], size=median_filter_kernel_size)

#diff = forw[indx] - forw_dilation
diff = forw[indx] - median_filtered_forw

fig, ax = plt.subplots(1, 5, figsize=(16, 8))
# ax[0].imshow(back, cmap='gray', vmin=0, vmax=255)
# ax[0].set_title('Backwards')
# ax[0].axis('off')

ax[0].imshow(forw[indx], cmap='gray', vmin=0, vmax=1)
ax[0].set_title('Forwards')
ax[0].axis('off')

ax[1].imshow(forw_erosion, cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Erosion')
ax[1].axis('off')

ax[2].imshow(forw_dilation, cmap='gray', vmin=0, vmax=1)
ax[2].set_title('Dilation')
ax[2].axis('off')

ax[3].imshow(median_filtered_forw, cmap='gray', vmin=0, vmax=1)
ax[3].set_title(f'Median Filter (Kernel: {median_filter_kernel_size})')
ax[3].axis('off')

ax[4].imshow(diff, cmap='gray', vmin=0, vmax=1)
ax[4].set_title('Difference (Forw - Dilation)')
ax[4].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
indx = 1000

fig, ax = plt.subplots(1, 2, figsize=(12, 6)) # Adjusted figure size for better display
ax[0].imshow(backward_patches[indx], cmap="viridis")
ax[0].set_title("Backward SHG")
ax[0].axis('off')

ax[1].imshow(backward_norm[indx], cmap="viridis", vmin=0, vmax=1)
ax[1].set_title("Backward SHG normalised")
ax[1].axis('off')

In [ ]:
import matplotlib.pyplot as plt
plt.subplot(2,2,1)
indx=1000
plt.imshow(backward_patches[indx], cmap="viridis")
plt.title("Backward SHG")

plt.subplot(2,2,2)
#img_adapteq_f = exposure.equalize_adapthist(forward_norm[0], clip_limit=0.2)
plt.imshow(forward_norm_1[indx], cmap='gray', vmin=0, vmax=1)
plt.title("Forward target")
plt.show()

plt.subplot(2,2,3)
plt.imshow(backward_norm[indx], cmap="viridis", vmin=0, vmax=1)
plt.title("Backward SHG normalised")

plt.subplot(2,2,4)
plt.imshow(forward_norm[indx], cmap='gray', vmin=0, vmax=1)
plt.title("Forward target")
plt.show()